In [5]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [6]:
import numpy as np

X_all = np.load(
    "/kaggle/input/datasets/mannangrover/diabetes-temporal-dataset/X_all.npy",
    mmap_mode="r"
)

y_all = np.load(
    "/kaggle/input/datasets/mannangrover/diabetes-temporal-dataset/y_all.npy"
)

participant_ids = np.load(
    "/kaggle/input/datasets/mannangrover/diabetes-temporal-dataset/participant_ids.npy"
)

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)
print("Participants:", len(np.unique(participant_ids)))

X shape: (1017314, 60, 6)
y shape: (1017314,)
Participants: 1533


In [7]:
from sklearn.model_selection import train_test_split
import numpy as np

unique_participants = np.unique(
    participant_ids
)

train_participants, test_participants = train_test_split(
    unique_participants,
    test_size=0.2,
    random_state=42
)

train_mask = np.isin(
    participant_ids,
    train_participants
)

test_mask = np.isin(
    participant_ids,
    test_participants
)

train_indices = np.where(
    train_mask
)[0]

test_indices = np.where(
    test_mask
)[0]

print("Train windows:", len(train_indices))
print("Test windows:", len(test_indices))

Train windows: 813128
Test windows: 204186


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class WearableDataset(Dataset):

    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        real_idx = self.indices[idx]

        return (
            torch.tensor(
                self.X[real_idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.y[real_idx],
                dtype=torch.float32
            )
        )

train_dataset = WearableDataset(
    X_all,
    y_all,
    train_indices
)

test_dataset = WearableDataset(
    X_all,
    y_all,
    test_indices
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2048,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Ready")

Ready


In [9]:
import torch.nn as nn

class LSTMClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=6,
            hidden_size=128,
            num_layers=2,
            dropout=0.3,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        _, (hidden, _) = self.lstm(x)

        x = hidden[-1]

        return self.fc(x).squeeze()

In [10]:
import numpy as np

device = torch.device("cuda")

class_counts = np.bincount(
    y_all[train_indices].astype(int)
)

print(class_counts)

negative = class_counts[0]
positive = class_counts[1]

pos_weight = torch.tensor(
    negative / positive,
    dtype=torch.float32
).to(device)

model = LSTMClassifier().to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print(model)

[280002 533126]
LSTMClassifier(
  (lstm): LSTM(6, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [11]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(
            device,
            non_blocking=True
        )

        batch_y = batch_y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(
            outputs,
            batch_y
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/10 Loss: 0.4723
Epoch 2/10 Loss: 0.4687
Epoch 3/10 Loss: 0.4656
Epoch 4/10 Loss: 0.4622
Epoch 5/10 Loss: 0.4590
Epoch 6/10 Loss: 0.4551
Epoch 7/10 Loss: 0.4519
Epoch 8/10 Loss: 0.4488
Epoch 9/10 Loss: 0.4445
Epoch 10/10 Loss: 0.4394


In [12]:
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(
            device,
            non_blocking=True
        )

        outputs = model(batch_X)

        probs = torch.sigmoid(outputs)

        preds = (
            probs > 0.5
        ).float()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            batch_y.numpy()
        )

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(
    "\nROC-AUC:",
    roc_auc_score(
        all_labels,
        all_probs
    )
)

print(
    "\nConfusion Matrix:\n",
    confusion_matrix(
        all_labels,
        all_preds
    )
)

print(
    "\nClassification Report:\n",
    classification_report(
        all_labels,
        all_preds
    )
)


ROC-AUC: 0.5543666181320195

Confusion Matrix:
 [[39961 33782]
 [58805 71638]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.40      0.54      0.46     73743
         1.0       0.68      0.55      0.61    130443

    accuracy                           0.55    204186
   macro avg       0.54      0.55      0.54    204186
weighted avg       0.58      0.55      0.56    204186

